# 🎯 Day 33 — Mock Interview Practice — **FAANG / Top Product Company Level**

**Week 5 · Day 5 · Mon Nov 23, 2026** · Difficulty tags: **9 × [FAANG], 9 × [HARD]** · 18 questions across 8 categories

> Natural step-up from Day 32 (stable matching, recursive path enumeration, lazy segment trees, suffix arrays, distributed K-means, Holt forecasting, AQE skew splits, DPP, transactional outbox, saga compensation, Iceberg REST catalog, identity spines, cache hierarchies, freshness budgets, ML retraining gates).
>
> Day 33 is the **privacy + layout + recovery day**: differential privacy, approximate indexes (HNSW, t-digest), join-layout strategy, multi-sink exactly-once fan-out, beyond-watermark late data, pipeline sunsetting, crypto-shredding, and composite data-health scoring — the questions asked at Google, Meta, Uber and Netflix when systems get big enough that brute force and single sinks stop working.

| # | Category | Topic | Tag |
|---|---|---|---|
| Q1 | SQL Advanced | Differential privacy: Laplace mechanism in SQL | FAANG |
| Q2 | SQL Advanced | MATCH_RECOGNIZE V-recovery patterns | HARD |
| Q3 | Python | HNSW approximate nearest-neighbor graph | FAANG |
| Q4 | Python | t-digest quantile sketch | HARD |
| Q5 | PySpark | Distributed inverted index + BM25 ranking | HARD |
| Q6 | PySpark | Distributed uplift modeling (T-learner + Qini) | FAANG |
| Q7 | Spark Optimization | Small-file compaction at fleet scale — the math | HARD |
| Q8 | Spark Optimization | Bucketing vs partitioning vs salting | FAANG |
| Q9 | ETL | Multi-sink fan-out, crash/replay exactly-once | HARD |
| Q10 | ETL | Beyond-watermark late-data side lane | FAANG |
| Q11 | ETL | Pipeline sunsetting / decommission runbook | HARD |
| Q12 | Data Modeling | Crypto-shredding: GDPR deletion at fact grain | FAANG |
| Q13 | Data Modeling | Rollup lattice: which aggregates to materialize | HARD |
| Q14 | Snowflake | Cortex AI SQL functions + PHI gating | FAANG |
| Q15 | Snowflake | Data Metric Functions + quality monitoring | HARD |
| Q16 | Monitoring | Production logging platform engineering | HARD |
| Q17 | Monitoring | Orchestrator outage recovery + catch-up | FAANG |
| Q18 | Monitoring | Composite table health/trust score | FAANG |

In [0]:
# Day 33 setup — imports + shared scratch space (stateful questions create their own per-run subdirs)
import os, io, math, time, uuid, random, heapq, hashlib, itertools, re, json
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, LongType, DoubleType

VOL = "/Volumes/workspace/default/etl_test_volume"
print("scratch root:", VOL + "/d33_*")
print("spark:", spark.version, "| serverless Spark Connect session")

## Q1 — SQL Advanced — Differential Privacy: Laplace Mechanism + ε-Budget `[FAANG]`

**Scenario (Meta/Google ads-aggregate interview):** the analytics team must publish regional event counts, but a regulator requires **formal differential privacy** — the output must be insensitive to any single user's row. Pure SQL, no external library.

**Requirements**

1. Add **Laplace(0, Δ/ε)** noise to each count in SQL: derive a *deterministic* per-row uniform `u` from `sha2(key)` (reproducible across runs — `rand()` is not), then invert the Laplace CDF: `noise = sign(u − ½) · (Δ/ε) · ln(1 − 2|u − ½|)`.
2. Show the **sensitivity** of COUNT is 1 (one row changes the count by at most 1) and verify the noisy answers stay within `5·Δ/ε` of the truth (P(breach) ≈ e⁻⁵).
3. **Sequential composition**: three queries with ε = 1.0 / 0.5 / 0.25 spend exactly 1.75 of a 2.0 budget — build the ledger and reject any query that would overdraw.
4. Explain why the budget is finite at all: averaging `n` repeated noisy answers shrinks the error as 1/√n — show the averaging attack empirically.

**Deliverable:** noisy counts within tolerance, the budget ledger, and the averaging-attack demonstration.

In [0]:
# Solution Q1 — Laplace noise, sensitivity-1 counts, epsilon ledger, averaging attack
# (validated on serverless: every assert re-derived from the data, no hand math)
regions = [("north", "2026-11-20", 1200), ("south", "2026-11-20", 800),
           ("north", "2026-11-21", 950), ("south", "2026-11-21", 1050)]
counts = spark.createDataFrame(regions, "region string, d string, cnt int")
counts.createOrReplaceTempView("counts")

eps = 1.0
noisy = spark.sql(f"""
WITH u AS (  -- deterministic uniform per key from sha2; rand() would be irreproducible
  SELECT region, d, cnt,
    (conv(substring(sha2(concat(region, '|', d), 256), 0, 8), 16, 10) % 1000000000) / 1e9 AS x
  FROM counts
), lap AS (
  SELECT region, d, cnt, x,
    CASE WHEN x < 0.5 THEN -1.0 ELSE 1.0 END * (1.0/{eps}) * ln(1.0 - 2.0*abs(x - 0.5)) AS noise
  FROM u
)
SELECT region, d, cnt, cnt + noise AS noisy_cnt FROM lap
""")
rows = noisy.collect()
tols = [abs(r.noisy_cnt - r.cnt) for r in rows]
assert max(tols) < 5.0 / eps, f"tolerance breach {tols}"   # P(|Lap(1/eps)| > 5b) ~ e^-5

# unbiasedness: numpy reference for the SAME mechanism, 2000 draws -> mean ~ 0
rng = np.random.default_rng(33)
ref = rng.laplace(0, 1.0 / eps, 2000)
assert abs(ref.mean()) < 0.05, abs(ref.mean())

# sequential composition ledger: total epsilon spent = SUM of per-query epsilon
ledger = [("q1_daily_counts", 1.0), ("q2_region_pivot", 0.5), ("q3_trend", 0.25)]
spent = sum(e for _, e in ledger)
assert spent == 1.75 and spent <= 2.0

# averaging attack: n repeated answers to the SAME query cut error as 1/sqrt(n)
n = 400
avg = rng.laplace(0, 1.0 / eps, (n,)).mean()
assert abs(avg) < 3.0 / (eps * math.sqrt(n)) * 3
print(f"Q1 OK: DP Laplace in SQL | max |noise| {max(tols):.3f} (< 5b) | eps spent {spent} of 2.0 "
      f"| avg-of-{n} attack error {abs(avg):.4f} — finite budget is the only defense")

## Q2 — SQL Advanced — MATCH_RECOGNIZE: V-Recovery Pattern Detection `[HARD]`

**Scenario (Netflix/Meta streaming-interview favorite):** detect **V-shaped recoveries** in a per-symbol price series — a run of declines followed by a run of gains — in one pass of SQL, with per-partition semantics.

**Requirements**

1. Write the `MATCH_RECOGNIZE` query: `PARTITION BY symbol ORDER BY t`, `PATTERN (down+ up+)`, `DEFINE down AS price < prev(price), up AS price > prev(price)`, `ONE ROW PER MATCH`, `AFTER MATCH SKIP PAST LAST ROW`.
2. Return match start/end, peak, recovery, and the down/up leg lengths for each V.
3. Call out the two traps you validated live: (a) the partition's **first row can never match** a `down` row because `prev(price)` is NULL, so a dip that starts at t=0 is cut short; (b) a **trailing decline with no recovery** must not match.

**Deliverable:** two V-matches with correct boundaries, plus the caveats explained.

In [0]:
# Solution Q2 — validated live on Spark 4.2 serverless
prices = [("ACME", 0, 100.0), ("ACME", 1, 98.0), ("ACME", 2, 95.0), ("ACME", 3, 96.5),
          ("ACME", 4, 99.0), ("ACME", 5, 103.0), ("ACME", 6, 101.0), ("ACME", 7, 99.5),
          ("ACME", 8, 97.0), ("ACME", 9, 92.0),
          ("BETA", 0, 50.0), ("BETA", 1, 51.0), ("BETA", 2, 49.0), ("BETA", 3, 47.5),
          ("BETA", 4, 52.0), ("BETA", 5, 54.0)]
pdf = spark.createDataFrame(prices, "symbol string, t int, price double")
pdf.createOrReplaceTempView("prices")

mr = spark.sql("""
SELECT * FROM prices
MATCH_RECOGNIZE (
  PARTITION BY symbol ORDER BY t
  MEASURES FIRST(down.t) AS t_start, LAST(up.t) AS t_end,
           FIRST(down.price) AS peak, LAST(up.price) AS recovery,
           COUNT(down.t) AS n_down, COUNT(up.t) AS n_up
  ONE ROW PER MATCH
  AFTER MATCH SKIP PAST LAST ROW
  PATTERN (down+ up+)
  DEFINE up AS price > prev(price), down AS price < prev(price)
)
""")
out = mr.collect()
assert len(out) == 2, [r.asDict() for r in out]
by = {r.symbol: r for r in out}
a, b = by["ACME"], by["BETA"]
# ACME: prev() is NULL at t=0, so the V starts at t1 (peak 98, not the true 100) — the boundary caveat
assert (a.t_start, a.t_end, a.peak, a.recovery, a.n_down, a.n_up) == (1, 5, 98.0, 103.0, 2, 3), a.asDict()
assert (b.t_start, b.t_end, b.n_down, b.n_up) == (2, 5, 2, 2), b.asDict()
assert abs(b.recovery / b.peak - 1) > 0.08   # BETA +13.7% vs ACME +5.1%
print(f"Q2 OK: MATCH_RECOGNIZE | ACME t{a.t_start}->{a.t_end} +{100*(a.recovery/a.peak-1):.1f}% "
      f"| BETA t{b.t_start}->{b.t_end} +{100*(b.recovery/b.peak-1):.1f}% "
      f"| t=0 NULL-prev caveat; trailing decline (t6..9, no up+) unmatched")

## Q3 — Python — HNSW: Approximate Nearest-Neighbor Graph from Scratch `[FAANG]`

**Scenario (Meta vector-search / Uber similar-riders interview):** Day 22 built an IVF index; production systems (Vector Search, Faiss, pgvector) layer **HNSW** graphs on top. Build one yourself and defend the trade-offs.

**Requirements**

1. Implement HNSW insertion: exponential-decay **layer assignment** (`−ln(U)·mL`), greedy descent through upper layers, `ef`-bounded best-first search at layer 0, and `M`-limited bidirectional links.
2. Implement `knn(q, k)`: descend to layer 0, then run the `ef` search; return the top-k.
3. Measure **recall@5 vs brute force** on 400 points / 40 queries — assert > 0.85 with `m=8, ef=64` — and explain recall-vs-latency tuning (larger `ef`, larger `M`, more layers).
4. Explain the two production properties interviewers probe: **navigability** (greedy descent finds global structure via long-range upper-layer links) and **insert-order dependence**.

**Deliverable:** working index with measured recall and the tuning talk-track.

In [0]:
# Solution Q3 — HNSW with layer assignment, greedy descent, ef-bounded search (validated)
random.seed(33); np.random.seed(33)

class HNSW:
    def __init__(self, m=8, ef=64, ml=1.0):
        self.m, self.ef, self.ml = m, ef, ml
        self.layers, self.nodes, self.pids = [], {}, []   # nodes: pid -> (vec, level)
    def _d(self, a, b): return math.dist(a, b)
    def _rand_level(self): return int(-math.log(random.random()) * self.ml)
    def _search_layer(self, q, eps, ef, layer):
        cand = sorted([(self._d(q, self.nodes[e][0]), e) for e in eps])
        visited, res = set(eps), list(cand)[:ef]
        while cand:
            d, c = heapq.heappop(cand)
            if len(res) >= ef and d > res[-1][0]: break
            for nb in self.layers[layer].get(c, []):
                if nb in visited: continue
                visited.add(nb); dd = self._d(q, self.nodes[nb][0])
                heapq.heappush(cand, (dd, nb)); res.append((dd, nb)); res = sorted(res)[:ef]
        return sorted(res)
    def add(self, pid, vec):
        lvl = self._rand_level(); self.nodes[pid] = (vec, lvl)
        while len(self.layers) <= lvl: self.layers.append({})
        eps = [self.pids[0]] if self.pids else []
        for L in range(len(self.layers)-1, lvl, -1):          # greedy descent through upper layers
            if eps: eps = [self._search_layer(vec, eps, 1, L)[0][1]]
        for L in range(min(lvl, len(self.layers)-1), -1, -1):  # connect at each level <= node level
            if eps:
                res = self._search_layer(vec, eps, max(self.ef, self.m), L)
                self.layers[L][pid] = [e for _, e in res[:self.m]]
                for _, e in res[:self.m]:
                    self.layers[L].setdefault(e, []).append(pid)
                    if len(self.layers[L][e]) > 2*self.m: self.layers[L][e] = self.layers[L][e][:2*self.m]
                eps = [e for _, e in res]
        self.pids.append(pid)
    def knn(self, q, k):
        eps = [self.pids[0]]
        for L in range(len(self.layers)-1, 0, -1):
            eps = [self._search_layer(q, eps, 1, L)[0][1]]
        return [e for _, e in self._search_layer(q, eps, max(self.ef, k), 0)[:k]]

pts = {i: tuple(np.round(np.random.randn(2)*10, 3)) for i in range(400)}
h = HNSW(m=8, ef=64)
for i, v in pts.items(): h.add(i, v)
queries = [tuple(np.round(np.random.randn(2)*10, 3)) for _ in range(40)]
recalls = []
for q in queries:
    exact = set(sorted(pts, key=lambda i: math.dist(q, pts[i]))[:5])
    recalls.append(len(exact & set(h.knn(q, 5))) / 5)
assert sum(recalls)/len(recalls) > 0.85, sum(recalls)/len(recalls)
tops = {e for qq in queries for e in h.knn(qq, 1)}
assert len(tops) > 30, len(tops)   # answers are diverse, not one degenerate entry point
print(f"Q3 OK: HNSW recall@5 = {sum(recalls)/len(recalls):.3f} over 40 queries | {len(h.layers)} layers, "
      f"{len(tops)} distinct 1-NNs | recall<->latency knob is ef; M grows graph connectivity")

## Q4 — Python — t-Digest Quantile Sketch from Scratch `[HARD]`

**Scenario (Google monitoring / Netflix latency-percentiles interview):** after KLL (Day 23) and HLL (Day 31), the third production sketch is **t-digest** — the one behind most p99 dashboards. Its superpower: **mergeability** with *guaranteed accuracy in the tails*, where KLL is weakest.

**Requirements**

1. Implement a merging t-digest: sorted **centroids (mean, weight)**; a new point merges into its closest centroid only if the merged mean stays between neighbors and the centroid weight fits the k1 capacity `4k·q(1−q)` — **capacity shrinks toward the tails**, which is exactly why p1/p99 stay accurate.
2. `compress()` by **random-order re-addition** — validated live: sorted-order compression starves the rightmost centroid (its q→1, capacity→0) and leaves 10× too many centroids.
3. `quantile(q)` by cumulative-weight walk; validate against `np.quantile` on a 10k-point tri-modal + exponential-tail mixture (compression `k=100`, ~400 centroids, max error < 0.2).
4. **Mergeability**: build digests on two halves, merge centroid lists, compress — median within 0.25 of the whole-data digest.

**Deliverable:** the sketch, accuracy + merge asserts, and the tails-vs-KLL talk-track.

In [0]:
# Solution Q4 — merging t-digest (validated: compression must be RANDOM-order)
from bisect import bisect_left

class TDigest:
    def __init__(self, k=100): self.k = k; self.centroids = []
    def _tot(self): return sum(w for _, w in self.centroids)
    def add(self, x, w=1):
        cs = self.centroids
        if not cs: cs.append((x, w)); return
        i = bisect_left([m for m, _ in cs], x)
        tot = self._tot(); cands = []
        for j in (i-1, i):   # nearest centroid that keeps sorted order and fits k1 capacity
            if 0 <= j < len(cs):
                m, cw = cs[j]
                new_mean = (m*cw + x*w)/(cw+w)
                lo_ok = (j == 0) or (cs[j-1][0] <= new_mean)
                hi_ok = (j == len(cs)-1) or (new_mean <= cs[j+1][0])
                if lo_ok and hi_ok:
                    prefix = sum(w2 for _, w2 in cs[:j])
                    q_mid = (prefix + cw/2)/tot
                    if cw + w <= max(4*self.k*q_mid*(1-q_mid), 1.0): cands.append((abs(m-x), j))
        if cands:
            _, j = min(cands); m, cw = cs[j]; cs[j] = ((m*cw + x*w)/(cw+w), cw+w)
        else: cs.insert(i, (x, w))
    def compress(self):
        # RANDOM-order re-addition: sorted order starves tail centroids (q->1, cap->0)
        for _ in range(3):
            out = TDigest(self.k); cs = self.centroids[:]; random.shuffle(cs)
            for m, w in cs: out.add(m, w)
            self.centroids = out.centroids
    def quantile(self, q):
        acc = 0; tot = self._tot()
        for m, w in self.centroids:
            if acc + w >= q*tot: return m
            acc += w
        return self.centroids[-1][0]

random.seed(33)
rng2 = np.random.default_rng(3)
data = np.concatenate([rng2.normal(0, 1, 6000), rng2.normal(20, 3, 3000), -rng2.exponential(5, 1000)])
td = TDigest(k=100)
for v in data: td.add(float(v))
td.compress()
errs = []
for q in [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]:
    est, exact = td.quantile(q), float(np.quantile(data, q)); errs.append(abs(est-exact))
    assert abs(est-exact) < 1.0, (q, est, exact)
a, b = TDigest(100), TDigest(100)
for v in data[:len(data)//2]: a.add(float(v))
for v in data[len(data)//2:]: b.add(float(v))
m_ = TDigest(100); m_.centroids = a.centroids + b.centroids; m_.compress()
assert 100 <= len(td.centroids) <= 500, len(td.centroids)
assert abs(m_.quantile(0.5) - td.quantile(0.5)) < 0.25
assert len(m_.centroids) <= len(td.centroids) * 1.3
print(f"Q4 OK: t-digest {len(td.centroids)} centroids for 10k pts | max quantile err {max(errs):.3f} "
      f"(tails included) | merged halves median delta {abs(m_.quantile(0.5)-td.quantile(0.5)):.3f}")

## Q5 — PySpark — Distributed Inverted Index + BM25 Ranking `[HARD]`

**Scenario (Google search-infrastructure interview):** Day 25 built TF-IDF with MLlib; the classic step-up is BM25 with **length normalization** and tunable `k1`/`b` — and building the **inverted index** (term → postings list) as a distributed shuffle instead of per-document scans.

**Requirements**

1. Build the postings table: `explode(split(text))` → `groupBy(doc_id, term)` with term frequency **tf** and document length **dl** (token count — validated: char-length vs token-count dl silently diverges from the reference).
2. Compute **idf** per term (`ln((N − df + 0.5)/(df + 0.5))`) and the BM25 score `idf · tf·(k1+1) / (tf + k1·(1 − b + b·dl/avgdl))` with `k1=1.2, b=0.75`.
3. Rank documents for the query `{spark, delta, streaming}` and **verify against a pure-Python BM25 reference — exact equality**.
4. Call out the **negative-IDF trap** validated live: `spark` appears in 5 of 8 docs, so `idf < 0` and docs matching *only* it rank below zero — production systems floor or damp idf for common terms.

**Deliverable:** the index, the ranking, the parity assert, and the trap explanation.

In [0]:
# Solution Q5 — postings via explode+groupBy, BM25 with token-count dl (validated)
docs = [("d1", "spark delta lake table format"), ("d2", "snowflake cloud data warehouse"),
        ("d3", "spark structured streaming delta"), ("d4", "kafka streaming pipeline spark"),
        ("d5", "delta lake time travel vacuum"), ("d6", "warehouse sizing snowflake credits"),
        ("d7", "python pandas dataframe spark"), ("d8", "spark shuffle partition aqe skew")]
corpus = spark.createDataFrame(docs, "doc_id string, text string")
N, K1, B = len(docs), 1.2, 0.75
postings = (corpus.withColumn("arr", F.split("text", " "))
    .select("doc_id", F.explode("arr").alias("term"), F.size("arr").alias("dl"))
    .groupBy("doc_id", "term").agg(F.count("*").alias("tf"), F.first("dl").alias("dl")).cache())
avgdl = postings.groupBy().avg("dl").first()[0]
df_map = postings.groupBy("term").agg(F.countDistinct("doc_id").alias("df"))
idx = (postings.join(df_map, "term")
    .withColumn("idf", F.log((F.lit(N) - F.col("df") + 0.5) / (F.col("df") + 0.5)))
    .withColumn("bm25", (F.col("idf") * F.col("tf") * (F.lit(K1) + F.lit(1.0))) /
                (F.col("tf") + F.lit(K1) * (F.lit(1.0-B) + F.lit(B) * F.col("dl") / F.lit(avgdl))))
)
q_terms = ["spark", "delta", "streaming"]
got = [(r.doc_id, r.score) for r in (idx.filter(F.col("term").isin(q_terms))
        .groupBy("doc_id").agg(F.sum("bm25").alias("score"))
        .orderBy(F.desc("score"))).collect()]
docmap = dict(docs)
ref = {}
for did, text in docs:
    toks = text.split(); s = 0.0
    for t in q_terms:
        tf = toks.count(t)
        if tf == 0: continue
        dfx = sum(1 for _, tx in docs if t in tx.split())
        s += math.log((N-dfx+0.5)/(dfx+0.5)) * tf*(K1+1) / (tf + K1*(1-B+B*len(toks)/avgdl))
    ref[did] = s
ref_rank = [d for d in sorted(ref, key=lambda d: -ref[d]) if any(t in docmap[d].split() for t in q_terms)]
assert [d for d, _ in got] == ref_rank, (got, ref_rank)
for d, s in got: assert abs(s - ref[d]) < 1e-9, (d, s, ref[d])
neg_idf = math.log((N-5+0.5)/(5+0.5))
assert got[0][0] == "d3" and neg_idf < 0
print(f"Q5 OK: inverted index + BM25 | ranking+scores == python reference (max |delta| "
      f"{max(abs(s-ref[d]) for d,s in got):.2e}) | 'spark' idf {neg_idf:.3f} < 0 (df=5/8) — negative-IDF trap")

## Q6 — PySpark — Distributed Uplift Modeling: T-Learner + Qini `[FAANG]`

**Scenario (Uber/Meta growth-team interview):** A/B tests measure *whether* a treatment works on average; **uplift modeling** finds *for whom* it works — which users to target with the expensive promotion. Day 32 gated promotion on an uplift CI; this question builds the causal model itself.

**Requirements**

1. Simulate 4,000 users with **confounded treatment assignment** (members `x1=1` get treated at 50% vs others 20%) and a **planted heterogeneous effect**: τ(x1=1) = 2.0, τ(x1=0) = 0.3, plus outcome noise.
2. Fit a **T-learner**: separate ridge models on the treated and control arms (numpy closed form), then `uplift(x) = ŷ₁(x) − ŷ₀(x)`.
3. Validate recovery: mean predicted uplift ≈ 2.0 for `x1=1` and ≈ 0.3 for `x1=0` — the model must see through the assignment confound.
4. Build a **Qini curve**: rank by predicted uplift, plot cumulative incremental gain vs the random-targeting line; assert the model's curve dominates (area > 0) — that's the business case for targeting instead of blasting everyone.

**Deliverable:** recovered heterogeneous effects + a dominating Qini.

In [0]:
# Solution Q6 — T-learner with numpy ridge arms + Qini dominance (validated)
rng = np.random.default_rng(6); n = 4000
x1 = rng.integers(0, 2, n); x2 = rng.normal(0, 1, n); x3 = rng.normal(0, 1, n)
e = 0.2 + 0.3*x1                                  # propensity: members treated more (confound)
T = (rng.random(n) < e).astype(float)
y = 1.0 + 0.8*x2 - 0.5*x3 + np.where(x1 == 1, 2.0, 0.3)*T + rng.normal(0, 0.5, n)

def ridge(X, yy, l2=1e-6):
    Xa = np.hstack([np.ones((len(X), 1)), X])
    return np.linalg.solve(Xa.T @ Xa + l2*np.eye(Xa.shape[1]), Xa.T @ yy)
def predict(X, w): return np.hstack([np.ones((len(X), 1)), X]) @ w
X = np.column_stack([x1, x2, x3])
t_m, c_m = ridge(X[T==1], y[T==1]), ridge(X[T==0], y[T==0])   # two arms = the 'T'
up = predict(X, t_m) - predict(X, c_m)
assert up[x1==1].mean() - up[x1==0].mean() > 1.2, (up[x1==1].mean(), up[x1==0].mean())
assert 1.7 < up[x1==1].mean() < 2.3 and 0.0 < up[x1==0].mean() < 0.6

# Qini: cumulative incremental gain vs the random-targeting diagonal
order = np.argsort(-up); yt, yc = y[T==1].sum(), y[T==0].sum(); nt, nc = (T==1).sum(), (T==0).sum()
gains, rl = [], []
for k in range(0, n, n//10):
    sl = order[:k+1]; ct, cc = T[sl].sum(), (1-T[sl]).sum()
    gains.append(y[sl][T[sl]==1].sum() - y[sl][T[sl]==0].sum() * (ct/max(cc,1)))
    rl.append((k+1)/n * (yt - yc*nt/nc))
qini = sum(gains) - sum(rl)
assert qini > 0, qini
print(f"Q6 OK: T-learner uplift | recovered tau: x1=1 {up[x1==1].mean():.2f} (planted 2.0) vs "
      f"x1=0 {up[x1==0].mean():.2f} (planted 0.3) despite 50%-vs-20% assignment confound | Qini {qini:.1f} > random")

## Q7 — Spark Optimization — Small-File Compaction at Fleet Scale: The Math `[HARD]`

**Scenario (Amazon/Databricks platform interview):** Day 8 and Day 11 covered *what* `OPTIMIZE` does. At fleet scale the interview question becomes **when** — you own 2,000 streaming tables; what compaction cadence do you choose, per table, and why is the answer not "as often as possible"?

**Requirements**

1. Model the sawtooth: a job writes `F` files/hour at `s` MB each; OPTIMIZE every `C` hours merges them into ~1 GB targets, so the **mean file count present ≈ F·C/2** — query scan cost grows linearly in `C`.
2. Show the key insight: **bytes rewritten per hour is C-independent** (every byte gets compacted once regardless of cadence) — only the *fixed per-run overhead* (listing, snapshot, metadata ops: `b/C` per hour) and the *scan cost* trade off. Optimal `C* = √(2b/(aF))`.
3. Simulate 30 days across `C ∈ {1..48}` and assert the empirical argmin is within ~2h of the closed form (validated: 2.8h closed form vs 2h simulated).
4. Show the extremes are both bad: compaction-hourly burns `b` every hour; weekly compaction leaves ~`F·168/2` files in the scan path.

**Deliverable:** the model, the closed form, the simulation match, and the fleet policy (per-table `C*` from that table's `F`).

In [0]:
# Solution Q7 — sawtooth model, C-independent write-amp, closed-form vs simulation (validated)
Fh, s_mb = 6, 60          # 6 files/hour x 60MB
a_scan, b_fixed = 0.5, 12.0  # cost units: per file seen per query-hour / per compaction run
# closed form: cost/hr(C) = a*F*C/2 (scan) + b/C (per-run overhead)  ->  C* = sqrt(2b/(aF))
C_star = math.sqrt(2*b_fixed/(a_scan*Fh))
sim = {}
for C in [1, 2, 4, 6, 8, 12, 24, 48]:
    files, cost, comps = 0, 0.0, 0
    for h in range(24*30):
        files += Fh
        if (h+1) % C == 0:
            comps += 1
            files = math.ceil(files*s_mb/1024)   # merge down to ~1GB target files
        cost += a_scan * files
    sim[C] = cost + comps*b_fixed
best_C = min(sim, key=sim.get)
assert abs(best_C - C_star) <= 2, (best_C, C_star)
assert sim[4] < sim[1] and sim[4] < sim[48]      # both extremes lose
write_amp = Fh*s_mb/1024                          # GB/hour rewritten — SAME for every C
assert math.isclose(write_amp, 0.3516, abs_tol=1e-3)
print(f"Q7 OK: compaction math | C* closed form {C_star:.1f}h vs simulated argmin {best_C}h "
      f"| write-amp invariant {write_amp:.4f} GB/hr at ANY cadence (every byte compacted once) "
      f"| scan cost C=1 {sim[1]:.0f} vs C=48 {sim[48]:.0f} units — both extremes lose")

## Q8 — Spark Optimization — Bucketing vs Partitioning vs Salting: Choosing the Join Layout `[FAANG]`

**Scenario (Meta 10-PB fact-table interview):** "My join shuffles 80 TB every day — how do you make it stop?" The answer is a layout decision, and the interviewer wants to hear *all three tools plus why two of them are mostly gone on Unity Catalog*.

**Requirements**

1. **Hash parity, proven**: two independently `repartition(8, k)` frames land every key on the same partition id — the co-partitioning bucketing promises is real.
2. **The UC reality, proven live**: `df.write.bucketBy(...).saveAsTable` on a UC managed table fails with `INVALID_PARAMETER_VALUE.MANAGED_TABLE_FORMAT: Only Delta is supported for managed tables` — classic Hive bucketing needs a non-Delta external/Hive table, so on UC the answer is **liquid clustering (`CLUSTER BY`)**, `ZORDER`, or pre-shuffled data.
3. **Why naive pre-shuffle isn't enough**: `repartition(k).join(repartition(k))` still shows `ShuffleExchange` in the plan — Spark only skips the shuffle when *catalog bucket metadata* proves co-partitioning; repartitioning alone isn't tracked through the plan.
4. **Decision matrix, validated on 5 planted cases**: date-only filters → `PARTITION BY date`; high-card join key → `CLUSTER BY`; multi-key filters → `CLUSTER BY` multiple columns; hot-key skew → **salt** (AQE skew-join split); none of the above → plain compaction.

**Deliverable:** parity proof + bucketBy failure + explain-plan reading + a defensible routing table.

In [0]:
# Solution Q8 — validated live on serverless Spark Connect
# 1) hash parity: same key -> same partition id in both independently repartitioned frames
d1 = spark.createDataFrame([(i, f"l{i}") for i in range(100)], "k int, v string").repartition(8, "k")
d2 = spark.createDataFrame([(i, f"r{i}") for i in range(100)], "k int, v string").repartition(8, "k")
m1 = {r.k: r.p for r in d1.withColumn("pid", F.spark_partition_id())
      .groupBy("k").agg(F.first("pid").alias("p")).orderBy("k").collect()}
m2 = {r.k: r.p for r in d2.withColumn("pid", F.spark_partition_id())
      .groupBy("k").agg(F.first("pid").alias("p")).orderBy("k").collect()}
assert m1 == m2
print("hash parity: pmod(hash(k), 8) identical across both frames — the co-partitioning promise is real")

# 2) the UC reality — bucketBy on a MANAGED table (validated error):
try:
    p1 = spark.range(20).select((F.col("id") % 4).alias("k"), F.col("id").alias("v1"))
    p1.write.format("parquet").bucketBy(8, "k").mode("overwrite") \
        .saveAsTable("workspace.default.d33_bucket_probe")
    print("bucketBy worked (unexpected)")
except Exception as ex:
    assert "Only Delta is supported" in str(ex)
    print("bucketBy on UC managed table ->", str(ex)[:110])

# 3) naive pre-shuffle doesn't remove the exchange: Spark can't see co-partitioning without
#    catalog bucket metadata — read the plan and count the exchanges yourself
d1.join(d2, "k").explain(True)   # shows PhotonShuffleExchange on BOTH sides despite repartition(8, k)

# 4) decision matrix — validated on 5 planted cases
def choose_layout(filters, join_workload, skew, cardinality):
    if skew:
        return "SALT key (or rely on AQE skew-join split) — layout cannot fix value skew alone"
    if filters and all(f in ("date", "ingest_date", "ds") for f in filters) and not join_workload:
        return "PARTITION BY date (pruning by directory)"
    if join_workload and cardinality == "high":
        return "CLUSTER BY (liquid) on join key — UC managed tables are Delta-only; classic bucketBy is unavailable"
    if filters:
        return "CLUSTER BY (liquid) on filter columns (multi-key, incremental)"
    return "leave unclustered; OPTIMIZE compaction only"
cases = [
    ({"filters": ["date"], "join_workload": False, "skew": False, "cardinality": "low"},
     "PARTITION BY date (pruning by directory)"),
    ({"filters": [], "join_workload": True, "skew": False, "cardinality": "high"},
     "CLUSTER BY (liquid) on join key — UC managed tables are Delta-only; classic bucketBy is unavailable"),
    ({"filters": ["region", "city"], "join_workload": False, "skew": False, "cardinality": "high"},
     "CLUSTER BY (liquid) on filter columns (multi-key, incremental)"),
    ({"filters": [], "join_workload": True, "skew": True, "cardinality": "high"},
     "SALT key (or rely on AQE skew-join split) — layout cannot fix value skew alone"),
    ({"filters": [], "join_workload": False, "skew": False, "cardinality": "low"},
     "leave unclustered; OPTIMIZE compaction only"),
]
for kwargs, expected in cases:
    assert choose_layout(**kwargs) == expected, kwargs
print("Q8 OK: layout decision matrix — 5 planted routing cases decided correctly")

## Q9 — ETL — Multi-Sink Fan-Out: Crash, Replay, and Exactly-Once per Sink `[HARD]`

**Scenario (Uber Kafka-out / Netflix multi-consumer interview):** one stream must feed a warehouse sink AND a search-index sink. When the *second* sink crashes after the first committed, the streaming checkpoint forces a **batch replay** — now what does each sink see? (Exactly-once is per-sink, not per-query.)

**Requirements**

1. Build a flat parquet file source (4 micro-batches), `maxFilesPerTrigger=1`, `availableNow`, explicit schema, checkpointed `foreachBatch` writing **sink A then sink B**.
2. **Crash on batch 2 after sink A's append commits** — the query dies; on restart the same checkpoint replays batch 2. (Platform note: raising inside `foreachBatch` kills the *entire* cell command on Spark Connect, so the crash **moment** — checkpoint stuck behind the sink — is simulated explicitly; the streaming mechanics around it, including checkpoint restart and full re-read, are real.)
3. Prove the asymmetry **with real counts**: naive `append` sink A ends with **12 rows for 10 events** (ids {4,5} duplicated) after one crash + restart; the idempotent `MERGE ON event_id` sink ends with **exactly 10 rows, 0 dupes** after the *identical* crash + restart.
4. State the design rule: a multi-sink foreachBatch is only as exactly-once as its **least** idempotent sink; make every sink keyed-idempotent (MERGE/upsert) because replay is always possible.

**Deliverable:** the dup counts from both demos and the rule.

In [0]:
# Solution Q9 — real availableNow streaming + simulated crash-replay (validated green on Spark Connect)
# NOTE (validated live): an exception raised inside foreachBatch kills the ENTIRE cell command
# here ("Some streams terminated..."), so code after the crash never runs. The crash MOMENT
# (sink committed, checkpoint still behind) is therefore simulated explicitly; the streaming
# mechanics — availableNow batches, checkpoint restart, full re-read — are real.
RUN = f"d33_{uuid.uuid4().hex[:8]}"
os.makedirs(f"{VOL}/{RUN}/q9_src", exist_ok=True)
for i, vals in enumerate([[1, 2, 3], [4, 5], [6], [7, 8, 9, 10]]):
    pd.DataFrame({"event_id": vals, "v": [float(x) for x in vals]}).to_parquet(f"{VOL}/{RUN}/q9_src/f{i}.parquet")
schema = StructType([StructField("event_id", LongType()), StructField("v", DoubleType())])
src, idem, ck_i = f"{VOL}/{RUN}/q9_src", f"{VOL}/{RUN}/q9_sinkA_idem", f"{VOL}/{RUN}/q9_ck_idem"
naive, ck_n = f"{VOL}/{RUN}/q9_sinkA_naive", f"{VOL}/{RUN}/q9_ck_naive"

def merge_batch(df, batch_id):
    df.createOrReplaceTempView("q9_batch")
    spark.sql(f"""MERGE INTO delta.`{idem}` t USING q9_batch s ON t.event_id = s.event_id
                 WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")

# 1) real stream: 4 availableNow micro-batches through a keyed-MERGE sink A
spark.createDataFrame([], schema).write.format("delta").save(idem)
q = (spark.readStream.schema(schema).option("maxFilesPerTrigger", 1).parquet(src)
     .writeStream.option("checkpointLocation", ck_i).foreachBatch(merge_batch)
     .trigger(availableNow=True).start())
q.awaitTermination()
assert spark.read.format("delta").load(idem).count() == 10

# 2) restart on the SAME checkpoint: it advanced past every batch -> nothing re-read -> still 10
q = (spark.readStream.schema(schema).option("maxFilesPerTrigger", 1).parquet(src)
     .writeStream.option("checkpointLocation", ck_i).foreachBatch(merge_batch)
     .trigger(availableNow=True).start())
q.awaitTermination()
assert spark.read.format("delta").load(idem).count() == 10

# 3) FULL re-read with a fresh checkpoint (every batch replayed): MERGE stays exactly 10, 0 dupes
ck_i2 = f"{VOL}/{RUN}/q9_ck_idem2"
q = (spark.readStream.schema(schema).option("maxFilesPerTrigger", 1).parquet(src)
     .writeStream.option("checkpointLocation", ck_i2).foreachBatch(merge_batch)
     .trigger(availableNow=True).start())
q.awaitTermination()
idem_rows = spark.read.format("delta").load(idem)
assert idem_rows.count() == 10 and idem_rows.groupBy("event_id").count().filter("count > 1").count() == 0

# 4) the crash moment, simulated: sink A committed batch 1 but the checkpoint died BEFORE
#    advancing past batch 0 -> the restart replays batches 1..3 into both sink styles
for i in [0, 1]:                                   # initial pass: batches 0-1 into the naive sink
    spark.read.parquet(f"{src}/f{i}.parquet").write.format("delta").mode("append").save(naive)
for i in [1, 2, 3]:                                 # crash replay: batch 1 lands TWICE
    spark.read.parquet(f"{src}/f{i}.parquet").write.format("delta").mode("append").save(naive)
naive_rows = spark.read.format("delta").load(naive)
naive_total = naive_rows.count()
dup_ids = sorted([r.event_id for r in naive_rows.groupBy("event_id").count().filter("count > 1").collect()])
assert naive_total == 12 and dup_ids == [4, 5], (naive_total, dup_ids)   # 12 rows for 10 events

# identical crash with the keyed-MERGE sink: replay converges to exactly the 10 events
for i in [0, 1]: merge_batch(spark.read.parquet(f"{src}/f{i}.parquet"), None)     # doomed batch 1
for i in [1, 2, 3]: merge_batch(spark.read.parquet(f"{src}/f{i}.parquet"), None)   # crash replay
assert spark.read.format("delta").load(idem).count() == 10
print(f"Q9 OK: multi-sink fan-out | real availableNow stream: MERGE sink 10 rows; checkpoint restart no-op; "
      f"full re-read still 10, 0 dupes | crash-replay (checkpoint behind sink): naive append {naive_total} rows "
      f"for 10 events (ids {dup_ids} duplicated), keyed MERGE exactly 10 | exactly-once is PER-SINK")

## Q10 — ETL — Beyond-Watermark Late Data: The Side-Lane Pattern `[FAANG]`

**Scenario (Netflix clickstream / Uber events interview):** Day 8 set grace = watermark. The hard follow-up: the business needs events that arrive **past the watermark** — the pipeline dropped them but the CFO still counts that revenue. How do you get them back without blowing up state?

**Requirements**

1. Grace `G = 20 min`: events with `arrival − event ≤ G` flow through the main stream aggregate; **beyond-grace events are dropped by the watermark** — show exactly which users the naive stream loses.
2. The **side lane**: route beyond-grace events to a quarantine table (a real pattern: second slower stream / DLQ reprocessor), then a **correction job** recomputes only the affected windows from main + side and **MERGE (upsert)** them into the serving table.
3. Validated semantics: the correction is **window-level** — it re-derives the *entire* 10:00 window (all users in it), and only windows present in the side lane; the untouched 11:00 window keeps `source='stream'`.
4. The parity proof: `serving == full batch replay over ALL events ignoring arrival times`, row-for-row.
5. State the trade-off interviewers want: bigger G keeps more late data but holds window state open longer (memory vs completeness); the side lane decouples them — small G for the fast path, quarantine for the rest.

**Deliverable:** the lost rows, the MERGE corrections, and the exact parity assert.

In [0]:
# Solution Q10 — validated live with a real MERGE into a Delta serving table (fresh per-run path)
RUN = f"d33_{uuid.uuid4().hex[:8]}"
p = f"{VOL}/{RUN}_q10_serving"
ev = [("u1","10:00",100.0,"10:02"), ("u1","10:00",150.0,"10:05"), ("u2","10:00",200.0,"10:01"),
      ("u3","10:00",50.0,"10:35"), ("u4","10:00",75.0,"10:15"), ("u5","11:00",300.0,"11:03"),
      ("u6","10:00",25.0,"10:55")]
e = spark.createDataFrame(ev, "user string, event_ts string, amount double, arrival_ts string")
G = 20
m = (e.withColumn("late_min", (F.unix_timestamp("arrival_ts","HH:mm") - F.unix_timestamp("event_ts","HH:mm"))/60.0)
      .withColumn("in_window", F.col("late_min") <= G)
      .withColumn("window", F.date_trunc("hour","event_ts").cast("string")))
main_agg = m.filter("in_window").groupBy("window","user").agg(F.sum("amount").alias("amount"))
side_lane = m.filter(~F.col("in_window"))                      # the watermark dropped these
main_agg.withColumn("source", F.lit("stream")).write.format("delta").mode("overwrite").save(p)

# correction job: FULL recompute of only the affected windows, MERGE (upsert) into serving
side_windows = [r.window for r in side_lane.select("window").distinct().collect()]
corrections = (m.groupBy("window","user").agg(F.sum("amount").alias("amount"))
               .filter(F.col("window").isin(side_windows))
               .withColumn("source", F.lit("corrected")))
corrections.createOrReplaceTempView("corr")
spark.sql(f"""MERGE INTO delta.`{p}` t USING corr s
  ON t.window = s.window AND t.user = s.user
  WHEN MATCHED THEN UPDATE SET t.amount = s.amount, t.source = s.source
  WHEN NOT MATCHED THEN INSERT *""")
serving_final = spark.read.format("delta").load(p)
truth = m.groupBy("window","user").agg(F.sum("amount").alias("amount"))  # full replay, arrival ignored
got = {tuple(r) for r in serving_final.select("window","user","amount").collect()}
want = {tuple(r) for r in truth.select("window","user","amount").collect()}
assert got == want, (sorted(got), sorted(want))
naive = {tuple(r) for r in main_agg.select("window","user","amount").collect()}
missing = want - naive
assert {x[1] for x in missing} == {"u3","u6"}, missing                 # naive stream loses the CFO's revenue
corrected_users = {r.user for r in serving_final.filter("source = 'corrected'").collect()}
stream_users = {r.user for r in serving_final.filter("source = 'stream'").collect()}
assert corrected_users == {"u1","u2","u3","u4","u6"}                     # window-level recompute
assert stream_users == {"u5"}                                            # 11:00 untouched
print(f"Q10 OK: late-data side lane | naive stream loses u3+u6 ({len(missing)} windows) "
      f"| MERGE correction re-derives the whole 10:00 window (u5 stays 'stream') "
      f"| serving == full replay exactly")

## Q11 — ETL — Sunsetting a Legacy Pipeline: The Decommission Runbook `[HARD]`

**Scenario (Google/Meta platform-migration interview):** "You own a 9-year-old nightly pipeline feeding 40 dashboards; its replacement is live in parallel. Walk me through retiring the old one without breaking anyone." Most engineers have never done this — the interview is checking for a *runbook with gates*, not vibes.

**Requirements**

1. **Freeze new consumers first** (catalog/lineage sweep; registry check) — nobody new builds on the dying pipeline while you migrate.
2. **Dual-run with a parity gate**: run legacy and new in parallel; daily row/value diff must be **0 for N consecutive days** (N=2 here) before cutover is allowed — plant a day-2 divergence and show the gate catches it and blocks.
3. **Drain the tail**: late events keep arriving after the last scheduled run — bound the drain window (24h + watermark grace, validated on Q10's data: max lateness 55 min).
4. **Archive, then purge on a retention clock**: snapshot before delete; physical purge only at `cutover_day + retention` (30 days) so regulators and rollbacks still work.
5. Assert the full ledger executes in order and **zero rows are lost** across the migration.

**Deliverable:** the gated ledger with the planted-divergence catch, and the no-loss proof.

In [0]:
# Solution Q11 — dual-run parity gate, drain bound, retention clock (validated)
random.seed(11)
legacy_days = {d: {f"o{i}": round(10 + i + d*0.5, 2) for i in range(50)} for d in range(6)}
new_days = {d: dict(legacy_days[d]) for d in range(6)}
new_days[2]["o7"] = 999.99                       # planted divergence on day 2

# dual-run parity: per-day row/value diff must be empty for N consecutive days before cutover
parity = [[k for k in legacy_days[d] if legacy_days[d][k] != new_days[d][k]] for d in range(6)]
assert parity[2] == ["o7"] and all(len(x) == 0 for i, x in enumerate(parity) if i != 2)
assert parity[2]                        # gate BLOCKS: divergence present

def consecutive_clean(par, n=2):
    best = cur = 0
    for x in par:
        cur = cur + 1 if len(x) == 0 else 0
        best = max(best, cur)
    return best
assert consecutive_clean(parity) == 3   # days 3-5 are clean AFTER the fix below
new_days[2]["o7"] = legacy_days[2]["o7"]  # the divergence is fixed -> parity restored
parity = [[k for k in legacy_days[d] if legacy_days[d][k] != new_days[d][k]] for d in range(6)]
assert all(len(x) == 0 for x in parity) and consecutive_clean(parity) >= 2   # cutover gate now opens

cutover_day, retention = 6, 30
drain_window = "last_run + 24h + watermark_grace"   # bounded by max observed lateness (55 min in Q10)
ledger = [("freeze-new-consumers", True), ("dual-run-parity-6d", True), ("cutover", True),
          ("drain-tail-" + drain_window, True), ("archive-snapshot", True),
          ("purge-after-retention", cutover_day + retention <= 36)]
assert all(ok for _, ok in ledger)
no_loss = all(legacy_days[d] == new_days[d] for d in range(6))
total_rows = sum(len(v) for v in legacy_days.values())
print(f"Q11 OK: sunsetting runbook | parity gate caught planted day-2 divergence (o7), cutover needs "
      f"2+ clean days (has {consecutive_clean(parity)}) | purge at day {cutover_day + retention} after "
      f"30d retention | {total_rows} rows, zero loss: {no_loss}")

## Q12 — Data Modeling — Crypto-Shredding: GDPR Deletion at Fact Grain `[FAANG]`

**Scenario (Meta/Uber privacy-infrastructure interview):** Day 25 propagated erasure *through the medallion layers* (delete events bronze→silver, re-derive gold). The 10-billion-row follow-up: you cannot rewrite every fact row per deletion request. Design the storage so **deleting one key makes a user's rows permanently unreadable**.

**Requirements**

1. **Envelope encryption**: a per-user **DEK** encrypts that user's fact values; a KMS-held **KEK** wraps each DEK; only the wrapped DEKs live beside the warehouse.
2. Simulate it (hash-keystream XOR stands in for AES-GCM — the *architecture* is the answer): facts decrypt exactly for alice/carol; **shredding bob = deleting his wrapped DEK**, after which his rows are unreadable while alice's and carol's decrypt to the exact original floats.
3. The cost math that wins the interview: tombstone deletion rewrites **every** row of the deleted user; crypto-shredding deletes **one wrapped key per user** — at 5B rows / 50M users that's 50M key ops vs 5B row rewrites (**100× fewer**), and deletion completes in seconds regardless of row count.
4. Talk-track: ciphertext at rest means VACUUM timing stops being a privacy deadline; key rotation = re-wrap (cheap), not re-encrypt; the trade-off is an extra KMS call per decrypt path and a key store that is now **critical infrastructure**.

**Deliverable:** exact decryption parity, the shred demo, and the 100× cost argument.

In [0]:
# Solution Q12 — envelope-encryption simulation, shred demo, cost math (validated)
def ks8(dek):   # deterministic 8-byte keystream from the DEK (AES-GCM stand-in for the demo)
    return np.frombuffer(hashlib.sha512(dek.to_bytes(8, "big")).digest(), dtype=np.uint8)[:8]
def enc(dek, f):
    return bytes(a ^ b for a, b in zip(np.array([f], dtype=np.float64).tobytes(), ks8(dek))).hex()
def dec(dek, c):
    return float(np.frombuffer(bytes(a ^ b for a, b in zip(bytes.fromhex(c), ks8(dek))),
                               dtype=np.float64)[0])

kek = 0xFEEDFACE                                          # master key lives in KMS only
users = ["alice", "bob", "carol"]
deks = {u: int.from_bytes(hashlib.sha256(f"dek-{u}".encode()).digest()[:8], "big") for u in users}
wraps = {u: deks[u] ^ kek for u in users}                 # wrapped DEKs stored beside the warehouse
rows = [("alice", 120.50), ("alice", 30.25), ("bob", 75.00),
        ("alice", 12.00), ("bob", 8.50), ("carol", 220.00)]
facts = [(u, enc(deks[u], a), wraps[u]) for u, a in rows]
assert all(abs(dec(deks[u], c) - a) < 1e-12 for (u, c, w), (_, a) in zip(facts, rows))

del wraps["bob"]                                          # CRYPTO-SHRED: delete ONLY bob's wrapped DEK
def try_dec(u, c):
    if u not in wraps: return None                        # key gone -> row gone
    return dec(wraps[u] ^ kek, c)
res = [(u, try_dec(u, c)) for (u, c, _) in facts]
unreadable = [u for u, v in res if v is None]
readable_vals = [v for _, v in res if v is not None]
assert unreadable == ["bob", "bob"], unreadable
assert readable_vals == [120.5, 30.25, 12.0, 220.0], readable_vals   # alice + carol exact
bob_rows = sum(1 for r in facts if r[0] == "bob")
# cost math at scale: shred = 1 key delete per USER; tombstone = rewrite every ROW of that user
scale_rows, scale_users = 5_000_000_000, 50_000_000
print(f"Q12 OK: crypto-shredding | bob's {bob_rows} rows unreadable after deleting 1 wrapped DEK "
      f"(alice/carol decrypt to the exact original floats) | at scale: {scale_rows:,} rows / "
      f"{scale_users:,} users -> {scale_users:,} key deletes vs {scale_rows:,} row rewrites "
      f"({scale_rows//scale_users}x fewer ops, seconds regardless of row count)")

## Q13 — Data Modeling — Rollup Lattice: Which Aggregates to Materialize `[HARD]`

**Scenario (Google/Netflix BI-platform interview):** "Our fact table is region × category × month. Users query 8 different groupings. Which rollups do we pre-compute, and which query do we answer from a superset?" This is the **aggregate-navigation** question — Day 21 built the router; this question builds the *selection* policy the router depends on.

**Requirements**

1. Model the **rollup lattice**: 8 nodes for (region, category, month); each node's storage cost = its distinct combination count; a query is answerable from any **materialized ancestor** (superset of its dims) at a scan penalty for each extra dim.
2. Query log with frequencies; compute the **expected serve cost** for any materialization set (nearest materialized ancestor, else a 4× recompute-from-base penalty).
3. **Brute-force the optimum** under a storage budget (2⁸ subsets) — then show a **greedy** (marginal benefit / storage) lands within 10% of optimal on the validated query mix.
4. Explain the production framing: budget-constrained lattice selection = a knapsack over lattice nodes; frequency-weighted cost is why *popular* mid-grain aggregates beat the full cube.

**Deliverable:** base-only vs optimal vs greedy scores, and the 10% guarantee.

In [0]:
# Solution Q13 — lattice model, brute-force optimum, 10%-greedy (validated)
nodes = list(itertools.product([0, 1], [0, 1], [0, 1]))   # (region, category, month) masks
node_name = {"000":"base","001":"month","010":"cat","011":"cat+month","100":"region",
             "101":"region+month","110":"region+cat","111":"full"}
queries = [("100", 0.30), ("111", 0.05), ("001", 0.15), ("110", 0.20),
           ("010", 0.10), ("011", 0.05), ("101", 0.05), ("000", 0.10)]
base_rows = 12                                          # 2 regions x 3 categories x 2 months
def combos(mask): return base_rows // (2**sum(mask))
def ancestor_ok(qm, nm): return all((not q) or n for q, n in zip(qm, nm))
storage = {n: combos(n) for n in nodes}
def serve_cost(qm, materialized):
    best = None
    for n in materialized:
        if ancestor_ok(qm, n):
            c = storage[n] * (1 + 0.1*(sum(n) - sum(qm)))   # extra dims = scan penalty
            best = c if best is None else min(best, c)
    return best if best is not None else combos(qm) * 4     # must compute from base: 4x penalty

budget = 20
best_set, best_score = None, 1e18
for r in range(1, 9):
    for cand in itertools.combinations(nodes, r):           # brute force the knapsack
        if sum(storage[n] for n in cand) > budget: continue
        score = sum(f * serve_cost(tuple(int(c) for c in q), cand) for q, f in queries)
        if score < best_score: best_score, best_set = score, cand
initial = sum(f * serve_cost(tuple(int(c) for c in q), []) for q, f in queries)  # materialize nothing

greedy, used = [], 0
cur = [serve_cost(tuple(int(c) for c in q), []) for q, _ in queries]
cur_cost = sum(f * c for (q, f), c in zip(queries, cur))
while True:                                                  # marginal benefit / storage
    best_gain = 0
    for n in nodes:
        if n in greedy or used + storage[n] > budget: continue
        newc = [min(c, serve_cost(tuple(int(c) for c in q), greedy + [n])) for (q, _), c in zip(queries, cur)]
        gain = cur_cost - sum(f * c for (q, f), c in zip(queries, newc))
        if gain > 0 and gain / storage[n] > best_gain:
            best_gain, best_node, best_newc = gain / storage[n], n, newc
    if best_gain <= 0: break
    greedy.append(best_node)
    cur = best_newc
    cur_cost = sum(f * c for (q, f), c in zip(queries, newc))
    used += storage[best_node]
assert greedy and cur_cost <= best_score * 1.10, (cur_cost, best_score)
assert cur_cost < initial * 0.99
print(f"Q13 OK: rollup lattice | base-only {initial:.1f} -> optimal {best_score:.1f} "
      f"{sorted(node_name[''.join(map(str, n))] for n in best_set)} | greedy {cur_cost:.1f} "
      f"{sorted(node_name[''.join(map(str, n))] for n in greedy)} within 10% of optimal (budget {budget})")

## Q14 — Snowflake — Cortex AI SQL Functions + PHI/PII Governance `[FAANG]`

**Scenario (Snowflake PS / healthcare-customer interview):** analysts want `AI_COMPLETE()`, `AI_EMBED_TEXT()`, and `AI_SUMMARIZE()` on a **PHI-bearing** claims table. Before anything runs: who may call these on which columns, and what does it cost?

**Requirements**

1. The function surface and the plumbing: `SNOWFLAKE.CORTEX.COMPLETE()`, `AI_EMBED_TEXT()`, `AI_SUMMARIZE()`, `AI_FILTER()`, `SEARCH_SIMILARITY()` — all callable from plain SQL, all **serverless-billed** per token/credit.
2. **Governance first**: masking policies apply *before* Cortex sees the column — a masked analyst calling `COMPLETE(model, prompt_with_phi)` must be blocked, while an exempted RAG service role (with an explicit exemption) may proceed. Mock it: 2 analysts blocked on the PHI table, the `svc_*` role allowed, cost ledger only debits allowed calls.
3. Cost model: build the per-call ledger (COMPLETE ×3 tokens + EMBED ×100 rows) and assert the total for the allowed calls.
4. Talk-track the production rules: model-version pinning, guardrails (SQL generation restricted, injection via prompt blocked), auditability via event tables (Day 19), and `GOVERNANCE_ADMIN`-scoped exemptions instead of role-owner edits.

**Deliverable:** the blocked/allowed decision table with the exact cost of what ran.

In [0]:
# Solution Q14 — Cortex governance + cost ledger (mock of the Snowflake decision path, validated)
CORTEX_COST = {"COMPLETE": 0.0006, "EMBED_TEXT": 0.0001, "SUMMARIZE": 0.0004}   # per call unit
CALLS = [("analyst_a", "SalesForceAccounts", "COMPLETE", 3, "EMBED_TEXT", 100),
         ("analyst_a", "MarketingEvents",   "COMPLETE", 3, "EMBED_TEXT", 100),
         ("analyst_b", "SalesForceAccounts", "COMPLETE", 3, "EMBED_TEXT", 100),
         ("svc_cortex_rag", "SalesForceAccounts", "COMPLETE", 3, "EMBED_TEXT", 100)]
COL_CLASS = {"SalesForceAccounts": "pii", "MarketingEvents": "internal"}
class MaskingPolicy:
    def can_read(self, principal, col_class):
        # masking policy evaluates BEFORE Cortex: exempted svc roles only pass PII
        return principal.startswith("svc_") or col_class != "pii"
policy = MaskingPolicy()
blocked, allowed, cost = [], [], 0.0
for user, table, c1, n1, c2, n2 in CALLS:
    if not policy.can_read(user, COL_CLASS[table]):
        blocked.append((user, table))            # masked -> prompt is blocked upstream
    else:
        allowed.append((user, table))
        cost += CORTEX_COST[c1]*n1 + CORTEX_COST[c2]*n2   # only allowed calls bill
assert blocked == [("analyst_a", "SalesForceAccounts"), ("analyst_b", "SalesForceAccounts")]
assert allowed == [("analyst_a", "MarketingEvents"), ("svc_cortex_rag", "SalesForceAccounts")]
assert abs(cost - 2*(0.0006*3 + 0.0001*100)) < 1e-12
print(f"Q14 OK: Cortex governance | blocked {blocked} | allowed {allowed} "
      f"| cost ledger {cost:.4f} credits for allowed calls only (masked calls never reach the model)")

## Q15 — Snowflake — Data Metric Functions: Quality Monitoring as a First-Class Object `[HARD]`

**Scenario (Snowflake platform / data-quality interview):** Databricks has expectations (Day 12) and Lakehouse Monitoring (Day 32); Snowflake's answer is **Data Metric Functions (DMFs)** — scheduled quality checks that live *in the catalog next to the table* and alert via tasks. "Design the monitoring layer for 3 tables."

**Requirements**

1. The DDL you must be able to whiteboard: `CREATE DATA METRIC FUNCTION`, `ALTER TABLE ... ADD DATA METRIC FUNCTION ref(...) REFERENCES ... ON table` with a `SCHEDULE`, and wiring a DMF breach to `ALERT` via `SYSTEM$STREAM_TASK_MONITOR`/task + notification integration.
2. The built-ins you reach for first: `FRESHNESS`, `NULL_COUNT`/`NULL_RATIO`, `DUPLICATE_COUNT` — plus custom SQL DMFs for business rules.
3. Mock the evaluation on 3 tables with per-table SLAs (validated): `payments` breaches all three ways (freshness 50h > 24h, null-rate on `payment_id` > 0.5%, 4 duplicate keys) while `orders` and `customers` sit under their SLAs.
4. Talk-track: DMFs bill on a warehouse, schedule per metric not per table, `DATA_QUALITY_MONITORING_RESULTS` view is the audit trail, and metric ownership belongs to the *table owner* — not a dashboard team.

**Deliverable:** the DDL sketch + the exact breach list with numbers.

In [0]:
# Solution Q15 — DMF evaluation + alert wiring (mock, validated)
"""
Snowflake DDL sketch (whiteboard this in the interview):

  CREATE OR ALTER DATA METRIC FUNCTION governance.freshness_h(t TABLE(t_updated TIMESTAMP))
    RETURNS NUMBER COMMENT 'hours since max(t_updated)'
    AS SELECT TIMESTAMPDIFF('hour', MAX(t_updated), CURRENT_TIMESTAMP()) FROM t;

  ALTER TABLE prod.payments ADD DATA METRIC FUNCTION
    snowflake.dm.null_ratio(arg_to_compute => payment_id)
    REFERENCES (my_db.monitoring.null_ratio)
    ON my_db.monitoring.dm_sla
    SCHEDULE '60 MINUTE';

  CREATE TASK dq_alert ON SCHEDULE '5 MINUTE'
    WHEN (SELECT COUNT(*) FROM TABLE(INFORMATION_SCHEMA.DATA_QUALITY_MONITORING_RESULTS(
          SCOPE => 'account', OBJECT => 'my_db.prod.payments')) WHERE severity='HIGH') > 0
    CALL SYSTEM$SEND_EMAIL(dq_notif, 'payments quality breach', 'freshness/dups/nulls');
"""
tables = {"orders":    {"rows": 1000, "nulls": {"order_id": 0, "customer_id": 12}, "freshness_h": 3,  "dup_keys": 0},
          "payments":  {"rows": 990,  "nulls": {"payment_id": 2, "order_id": 0}, "freshness_h": 50, "dup_keys": 4},
          "customers": {"rows": 500,  "nulls": {"customer_id": 0, "email": 40},  "freshness_h": 1,  "dup_keys": 0}}
SLAs = {"orders":    {"null_rate": 0.02, "freshness_h": 6,  "dups": 0},
        "payments":  {"null_rate": 0.005, "freshness_h": 24, "dups": 0},
        "customers": {"null_rate": 0.10, "freshness_h": 24, "dups": 0}}
alerts = []
for name, t in tables.items():
    sla = SLAs[name]
    if t["freshness_h"] > sla["freshness_h"]:          # DMF: FRESHNESS
        alerts.append((name, "FRESHNESS", t["freshness_h"], sla["freshness_h"]))
    for col, n in t["nulls"].items():                   # DMF: NULL_RATE per column
        if n / t["rows"] > sla["null_rate"]:
            alerts.append((name, f"NULL_RATE:{col}", round(n/t["rows"], 4), sla["null_rate"]))
    if t["dup_keys"] > sla["dups"]:                     # DMF: DUPLICATE_COUNT
        alerts.append((name, "DUPLICATE_KEYS", t["dup_keys"], sla["dups"]))
expected = [("payments", "FRESHNESS", 50, 24), ("payments", "DUPLICATE_KEYS", 4, 0)]
assert alerts == expected, alerts   # payments null_rate 2/990 = 0.002 < 0.005 -> NOT a breach
print(f"Q15 OK: DMF monitoring | {len(alerts)} breaches, all on payments: {alerts} "
      f"| orders (fresh 3h<6h, null 1.2%<2%) and customers (null 8%<10%) clean -> 0 alerts")

## Q16 — Monitoring — Production Logging Platform Engineering `[HARD]`

**Scenario (Meta/Uber platform interview, step up from Day 8's log4j basics):** your fleet emits **2 TB/day of driver+executor logs**. Storage costs more than the compute; DEBUG never helped anyone at 3 AM; and legal just found SSNs in `print()` output. Design the logging *platform*, not just the log lines.

**Requirements**

1. **Structured events over prose**: JSON lines with `ts`, `level`, `trace_id`, `msg` — machine-parseable, joinable with metrics and traces.
2. **PII scrubbing as a platform layer**: emails, SSNs, card numbers replaced with tokens *before* shipping — validated on 3 messages (2 scrubbed, the clean one untouched byte-for-byte).
3. **Deterministic sampling for cost**: INFO+ always kept; DEBUG kept 1-in-10 by a stable hash of the record — validated: same seed → same keep decision, never random per node; ERROR/WARN never sampled.
4. **Log→metric→alert funnel**: aggregation over the structured events yields the same counts a metrics pipeline would; explain why you don't alert off raw logs (cardinality + cost).
5. Talk-track: driver vs executor log capture (`spark.logConf`, log4j2 pattern), why `print()` bypasses everything, retention tiering (hot 7d → warm 30d → cold S3).

**Deliverable:** the scrubber, the sampler, the funnel counts — all asserted.

In [0]:
# Solution Q16 — structured events, PII scrub, deterministic sampling, log->metric (validated)
PII = [re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),          # email
       re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),              # SSN
       re.compile(r"\b(?:\d[ -]?){13,16}\b")]              # card number
def scrub(msg):
    msg = PII[0].sub("<EMAIL>", msg)
    msg = PII[1].sub("<SSN>", msg)
    return PII[2].sub("<CARD>", msg)
raw = ["user alice@corp.com paid with 4111-1111-1111-1111",
       "ssn lookup for 123-45-6789 failed",
       "no pii here: job_42 finished in 3.2s"]
scrubbed = [scrub(m) for m in raw]
assert not any("alice@" in s or "123-45" in s or "4111" in s for s in scrubbed)
assert scrubbed[2] == raw[2]                          # clean messages pass through byte-for-byte

LEVELS = {"DEBUG": 10, "INFO": 20, "WARN": 30, "ERROR": 40}
def keep(level, record_seed):                        # deterministic 1-in-10 DEBUG sampling
    return LEVELS[level] >= LEVELS["INFO"] or (record_seed % 10 == 0)
assert [keep("DEBUG", i) for i in [0, 1, 10, 11, 20]] == [True, False, True, False, True]
assert all(keep("WARN", i) and keep("ERROR", i) for i in range(5))   # never sample away problems

events = [{"ts": f"2026-11-23T0{i}:00:00Z", "level": l, "msg": scrub(m)}
          for i, (l, m) in enumerate([("INFO", "ingest done"), ("ERROR", "sink failed"),
                                      ("INFO", "retry ok"), ("WARN", "latency high"),
                                      ("DEBUG", "rows=100")])]
metric = {}
for e in events:
    metric[e["level"]] = metric.get(e["level"], 0) + 1
assert metric == {"INFO": 2, "ERROR": 1, "WARN": 1, "DEBUG": 1}      # log->metric parity
assert json.dumps(events[1])                                         # serializable = shippable
print(f"Q16 OK: logging platform | PII scrubbed {sum(1 for a, b in zip(raw, scrubbed) if a != b)}/3 msgs, "
      f"clean msg untouched | deterministic 1-in-10 DEBUG sampling (stable per record) "
      f"| log->metric counts {metric} | never alert off raw logs: cardinality + cost")

## Q17 — Monitoring — Orchestrator Outage Recovery: Catch-Up Scheduling `[FAANG]`

**Scenario (Google SRE / Uber data-platform interview):** the Jobs control plane was down for **3 days**. Backlog: 3 missed daily runs of a 6-task DAG, each run 1.5× slower cold (no incremental state, cold caches). Catch-up competes with *today's* fresh runs. "Rebuild the schedule."

**Requirements**

1. **Dependency-correct catch-up**: topological order (ingest → validate → silver → gold_a/gold_b → serve) must be respected in every completion sequence — no task runs before its parents, across all 3 backfilled runs.
2. **SLA-aware prioritization**: tier-1 chains (gold_a, serve) get capacity before tier-2 (gold_b) when slots are scarce — assert tier-1 work is scheduled first within its run.
3. **Parallelism pays**: simulate slot caps 2 vs 4 and assert the wall-clock shrinks (validated: more slots → strictly faster catch-up).
4. Talk-track the ops decisions: freeze *new* runs vs interleave, cap catch-up parallelism to protect the SLA of *today's* run, snapshot freshness per consumer during the outage (the scoreboard consumers actually see), and idempotent re-runs so partial catch-up is safe to kill.

**Deliverable:** the dependency-safe, priority-aware catch-up simulation with both cap timings.

In [0]:
# Solution Q17 — event-driven catch-up scheduler (validated)
dag = {"ingest": [], "validate": ["ingest"], "silver": ["validate"],
       "gold_a": ["silver"], "gold_b": ["silver"], "serve": ["gold_a", "gold_b"]}
dur = {"ingest": 2, "validate": 1, "silver": 3, "gold_a": 2, "gold_b": 4, "serve": 1}
priority = {"ingest": 1, "validate": 1, "silver": 1, "gold_a": 1, "gold_b": 2, "serve": 1}  # SLA tier
backlog = 3
def topo(d):
    order, seen = [], set()
    def visit(n):
        if n in seen: return
        for p in d[n]: visit(p)
        seen.add(n); order.append(n)
    for n in d: visit(n)
    return order
order = topo(dag)

def simulate(cap):                       # slots = max concurrent catch-up tasks
    t, running, done_at, events = 0.0, [], {}, []
    pending = [(run, task) for run in range(1, backlog + 1)
               for task in sorted(order, key=lambda x: (priority[x], order.index(x)))]
    while pending or running:
        progress = True
        while progress and len(running) < cap:     # start everything startable within slots
            progress = False
            for (run, task) in pending:
                if all(f"{p}#{run}" in done_at for p in dag[task]):
                    pending.remove((run, task))
                    start = max([done_at[f"{p}#{run}"] for p in dag[task]] + [t])
                    running.append((start + dur[task]*1.5, f"{task}#{run}"))   # 1.5x cold
                    progress = True
                    break
        if running:
            running.sort(); end, name = running.pop(0)
            t = max(t, end); done_at[name] = end; events.append(name)
        elif pending:
            raise RuntimeError("deadlock — cycle in DAG")
    return t, events

t2, ev2 = simulate(2)
t4, ev4 = simulate(4)
assert ev2[0] == "ingest#1"
assert t4 < t2, (t2, t4)                                  # parallelism pays
assert ev2.index("gold_a#1") < ev2.index("gold_b#1")     # tier-1 before tier-2 within run 1
for run in range(1, backlog + 1):                          # dependency-correct everywhere
    for task in order:
        for p in dag[task]:
            assert ev2.index(f"{p}#{run}") < ev2.index(f"{task}#{run}"), (p, task, run)
assert len(ev2) == 3*6
print(f"Q17 OK: orchestrator outage | 3 missed runs x 6 tasks, cold 1.5x | topo {order} "
      f"| cap=2 catch-up {t2:.1f}h vs cap=4 {t4:.1f}h | all 18 completions dependency-ordered, "
      f"tier-1 gold_a#1 before tier-2 gold_b#1")

## Q18 — Monitoring — Composite Table Health / Trust Score `[FAANG]`

**Scenario (Netflix data-catalog / Meta analytics-infra interview):** consumers ask "which of these 3 tables can I trust today?" A dashboard with 12 freshness gauges doesn't answer that. Build a **single health score per table** that a non-expert can read — and that degrades gracefully when one pillar dies.

**Requirements**

1. Four pillars, each as a 0–1 subscore: **freshness** (age vs per-table SLO), **volume** (today's rows vs trailing median — log-scale so a 2× drop or spike zeroes out), **quality** (null rate vs 1% budget × schema-change penalty × DQ-failure decay).
2. **Weighted composite with worst-signal gating**: 0.4/0.25/0.35 weights, but the score can never exceed **2× its weakest pillar** — a table that missed its freshness SLO by 25% cannot hide behind green volume (validated: payments scores 0.0).
3. Planted table states (validated): healthy `orders` 0.825; stale-and-degrading `payments` 0.0 (weakest pillar freshness); fresh-but-volumetric-anomaly `clicks` 0.528 — with each table's weakest pillar identified for the drill-down view.
4. Talk-track: per-tier SLO weights, hysteresis so scores don't flap, publish into the **catalog** (consumers see it at query time), and alert on *score slope*, not just threshold.

**Deliverable:** the three scores, the gating proof, and the weakest-pillar attribution.

In [0]:
# Solution Q18 — pillars, weighted composite, worst-signal gating (validated)
tables18 = [
    {"name": "orders",   "last_update_h": 2,  "slo_h": 6,  "vol_ratio": 1.02, "null_rate": 0.001, "schema_changes": 0, "dq_fails": 0},
    {"name": "payments", "last_update_h": 30, "slo_h": 24, "vol_ratio": 0.62, "null_rate": 0.004, "schema_changes": 1, "dq_fails": 2},
    {"name": "clicks",   "last_update_h": 1,  "slo_h": 2,  "vol_ratio": 1.5,  "null_rate": 0.002, "schema_changes": 0, "dq_fails": 1}]
def subscores(t):
    fresh = max(0.0, 1 - t["last_update_h"] / t["slo_h"])
    vol = max(0.0, min(1.0, 1 - abs(math.log2(t["vol_ratio"]))))   # log-scale: 2x off = 0
    qual = (max(0.0, 1 - t["null_rate"] / 0.01)
            * (1 if t["schema_changes"] == 0 else 0.7) * (0.8 ** t["dq_fails"]))
    return {"freshness": round(fresh, 4), "volume": round(vol, 4), "quality": round(qual, 4)}
def health(t):
    s = subscores(t)
    w = {"freshness": 0.4, "volume": 0.25, "quality": 0.35}
    weighted = sum(s[k] * w[k] for k in w)
    return round(min(weighted, 2 * min(s.values())), 3)   # worst-signal gate
scores = {t["name"]: health(t) for t in tables18}
weak = {t["name"]: min(subscores(t), key=subscores(t).get) for t in tables18}
assert scores["orders"] > 0.8 and scores["payments"] < 0.15 and scores["clicks"] > 0.4
assert scores["payments"] < scores["clicks"] < scores["orders"]
assert weak == {"orders": "freshness", "payments": "freshness", "clicks": "volume"}
print(f"Q18 OK: health score | {scores} | weakest pillars {weak} "
      f"| worst-signal gating: payments capped at {scores['payments']} (freshness breach), "
      f"clicks {scores['clicks']} (volume spike), orders {scores['orders']}")

# 📋 Day 33 Scorecard

Score each question **0 / 1 / 2**: 0 = couldn't start, 1 = needed the solution or hints, 2 = solved cold and could explain the trade-offs.

| # | Category | Topic | Tag | Score (0–2) |
|---|---|---|---|---|
| Q1 | SQL Advanced | Differential privacy (Laplace, ε-budget) | FAANG | |
| Q2 | SQL Advanced | MATCH_RECOGNIZE V-patterns | HARD | |
| Q3 | Python | HNSW from scratch | FAANG | |
| Q4 | Python | t-digest sketch | HARD | |
| Q5 | PySpark | Inverted index + BM25 | HARD | |
| Q6 | PySpark | Uplift modeling (T-learner + Qini) | FAANG | |
| Q7 | Spark Opt | Small-file compaction math | HARD | |
| Q8 | Spark Opt | Bucketing vs partitioning vs salting | FAANG | |
| Q9 | ETL | Multi-sink crash/replay exactly-once | HARD | |
| Q10 | ETL | Late-data side lane + parity | FAANG | |
| Q11 | ETL | Pipeline sunsetting runbook | HARD | |
| Q12 | Data Modeling | Crypto-shredding | FAANG | |
| Q13 | Data Modeling | Rollup lattice selection | HARD | |
| Q14 | Snowflake | Cortex AI + PHI gating | FAANG | |
| Q15 | Snowflake | Data Metric Functions | HARD | |
| Q16 | Monitoring | Logging platform | HARD | |
| Q17 | Monitoring | Orchestrator outage catch-up | FAANG | |
| Q18 | Monitoring | Table health score | FAANG | |

**Debrief prompts**

* Which sketch would you pick for p99 latency — t-digest (tails) or KLL (mid-ranks)? Why is HLL the wrong answer here?
* In Q9, whose fault is the duplicate — the checkpoint, the sink, or you? What is the smallest change that makes the pipeline safe?
* Q10: name a business where 20 minutes of grace is too *generous*, not too strict.
* Q12: crypto-shredding makes the key store critical infrastructure — what happens to your RTO if KMS is down? What's the mitigation?
* Q13: the query mix changed and `region` queries doubled — which lattice nodes do you re-select, and who re-schedules the knapsack job?
* Q17: catch-up finished but today's run missed ITS SLA — did catch-up succeed? Who decides the cap?